In [ ]:
%load_ext autoreload
%autoreload 2
from machine_lib import * 
from mylib import *
# TSOps,
s = login()


In [35]:
# 获取数据集准备喂给gpt
# 获取数据集准备喂给gpt
df = get_datafields(s, dataset_id = 'fundamental94', region='KOR', universe='TOP600', delay=1)


In [ ]:
n=df.id+":"+df.description
print(','.join(n))

In [ ]:
import pandas as pd
# 树表转换
# 读取 Excel 文件
file_path = "押品类目.xlsx"  # 替换为您的文件路径
data = pd.read_excel(file_path, header=None)

# 添加列名，假设 A 列为 "编号"，B 列为 "名称"
data.columns = ["编号", "名称"]

# 转换编号为字符串，避免整数处理导致的问题
data["编号"] = data["编号"].astype(str)

# 定义菜单生成逻辑
result = []

for _, row in data.iterrows():
    code = row["编号"].strip()  # 编号（移除可能的多余空格）
    name = row["名称"].strip()  # 名称（移除可能的多余空格）

    # 一级菜单
    if len(code) == 2:
        level_1 = name
        result.append([level_1, "", "", ""])

    # 二级菜单
    elif len(code) == 5:
        try:
            parent_code = code[:2]  # 前两位对应的一级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = name
            result.append([level_1, level_2, "", ""])
        except IndexError:
            print(f"Warning: 找不到一级菜单对应编号 {parent_code}")

    # 三级菜单
    elif len(code) == 8:
        try:
            parent_code = code[:2]  # 前两位对应的一级编号
            child_code = code[:5]  # 前五位对应的二级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = data.loc[data["编号"] == child_code, "名称"].values[0]
            level_3 = name
            result.append([level_1, level_2, level_3, ""])
        except IndexError:
            print(f"Warning: 找不到二级菜单或一级菜单对应编号 {child_code} 或 {parent_code}")

    # 四级菜单
    elif len(code) == 11:
        try:
            # print(1)
            # if code=='4008011010': 
            #     print(1)
            parent_code = code[:2]      # 前两位对应的一级编号
            child_code = code[:5]      # 前五位对应的二级编号
            sub_child_code = code[:8]  # 前十位对应的三级编号
            level_1 = data.loc[data["编号"] == parent_code, "名称"].values[0]
            level_2 = data.loc[data["编号"] == child_code, "名称"].values[0]
            level_3 = data.loc[data["编号"] == sub_child_code, "名称"].values[0]
            level_4 = name
            if level_4:  # 仅当四级菜单不为空时才加入
                result.append([level_1, level_2, level_3, level_4])
        except IndexError:
            print(f"Warning: 找不到三级菜单、二级菜单或一级菜单对应编号 {sub_child_code}、{child_code} 或 {parent_code}")
    else:
        print(code)

# 转换结果为 DataFrame
result_df = pd.DataFrame(result, columns=["一级菜单", "二级菜单", "三级菜单", "四级菜单"])

# 删除四级菜单为空的行
result_df = result_df[result_df["四级菜单"].notna() & (result_df["四级菜单"] != "")]

# 导出为 Excel 文件
result_df.to_excel("四级菜单转换结果_过滤空行.xlsx", index=False)

print("转换完成，结果已保存为四级菜单转换结果_过滤空行.xlsx")


In [ ]:
import openpyxl
# 拆分合并单元格
# 加载工作簿
wb = openpyxl.load_workbook('押品分类比对.xlsx')

# 选择要处理的工作表
sheet = wb.active  # 或者 wb['Sheet1']

# 将合并单元格范围复制到一个新的列表中
merged_ranges = list(sheet.merged_cells.ranges)

# 遍历合并单元格的所有范围
for merged_range in merged_ranges:
    # 获取合并单元格的范围的边界 (min_col, min_row, max_col, max_row)
    min_col, min_row, max_col, max_row = merged_range.bounds
    
    # 获取合并单元格左上角单元格的值
    start_cell = sheet.cell(row=min_row, column=min_col)
    value = start_cell.value  # 获取该坐标处单元格的值
    
    # 确保值是字符串类型，避免出现 'Cell' 对象的问题
    if value is not None:
        value = str(value)  # 转换为字符串类型
    
    # 拆分合并单元格
    sheet.unmerge_cells(str(merged_range))  # 拆分合并单元格
    
    # 填充拆分后的每个单元格的值
    for row in range(min_row, max_row + 1):
        for col in range(min_col, max_col + 1):
            cell = sheet.cell(row=row, column=col)
            cell.value = value  # 将值填充到每个单元格

# 保存修改后的工作簿
wb.save('your_file_modified.xlsx')


ts_mean, ts_median, ts_max, ts_min, ts_sum, ts_std_dev, ts_returns, ts_scale, ts_max_diff, ts_min_diff, ts_rank, ts_corr, ts_covariance, ts_partial_corr, ts_product, ts_decay_exp_window, ts_decay_linear, ts_zscore, ts_entropy


In [ ]:
ops='ts_mean, ts_median, ts_max, ts_min, ts_sum, ts_std_dev, ts_returns, ts_scale, ts_max_diff, ts_min_diff, ts_rank, ts_corr, ts_covariance, ts_partial_corr, ts_product, ts_decay_exp_window, ts_decay_linear, ts_zscore, ts_entropy'.split(',')
profit='fnd94_is_gross_inc_q, fnd94_is_oper_inc_q, fnd94_is_net_inc_basic_q, fnd94_is_net_inc_dil_q, fnd94_is_eps_basic_q, fnd94_is_eps_contin_oper_q, fnd94_is_ebit_oper_q, fnd94_is_ebitda_q'.split(',')
asset='fnd94_bs_assets_tot_q, fnd94_bs_eq_tot_q, fnd94_bs_debt_q, fnd94_bs_liabs_tot_q, fnd94_bs_stock_com_q, fnd94_bs_com_eq_apic_q, fnd94_bs_com_eq_retain_earn_q, fnd94_bs_com_eq_for_exch_q, fnd94_bs_com_eq_par_q, fnd94_bs_treas_stk_q, fnd94_des_mkt_cap_q, fnd94_des_net_debt_q, fnd94_rt_com_eq_assets_q, fnd94_rt_debt_com_eq_q, fnd94_rt_assets_com_eq_q'.split(',')
days='30,60,120,250,360'.split(',')

In [ ]:
expStr='A(B/C, D)'
exps=[]
for A in ops:
    for B in profit:
        for C in asset:
            for D in days:
                exps.append(expStr.replace('A',A).replace('B',B).replace('C',C).replace('D',D)) 

                       

In [ ]:
so_tracker = get_alphas("11-19", "11-24", 1.2, 0.8, "KOR", 100, "track")


In [ ]:
so_alpha_list=[]
region='KOR'
group_ops = ["group_neutralize", "group_rank", "group_normalize", "group_scale", "group_zscore"]

for t in so_tracker['next']:
        for alpha in get_group_second_order_factory([t[1]], group_ops, region):
            so_alpha_list.append((alpha,t[7]))


In [ ]:
print(len(so_alpha_list))

In [ ]:
so_pools = load_task_pool(so_alpha_list, 10, 10)


In [ ]:
multi_simulate(so_pools, 'MARKET', 'KOR', 'TOP600', 43)


In [ ]:
# 1.58 sharpe, 1 fitness, "submit"参数
th_tracker = get_alphas("10-15", "11-27", 1.58, 1, "KOR", 5000, "submit")
print(len(th_tracker))

In [ ]:
print(len(th_tracker))

In [ ]:
## 将get的alpha的id取出至stone_bag，用api check submission
stone_bag = []
for alpha in th_tracker['next'] + th_tracker['decay']:
    stone_bag.append(alpha[0])
print(len(stone_bag))
gold_bag = []
check_submission(stone_bag, gold_bag, 325)

In [ ]:
check_submission(stone_bag, gold_bag, 2245)

In [ ]:
gold_bag

[('', np.float64(0.5981)),
 ('', np.float64(0.5867)),
 ('', np.float64(0.4892)),
 ('', np.float64(0.6144)),
 ('', np.float64(0.655)),
 ('wEPLnAQ', np.float64(0.478)),--
 ('wEPLnJd', np.float64(0.4493)),x
 ('MLZkb8r', np.float64(0.4584)),x
 ('Q9gEnjK', np.float64(0.4846)),
 ('PO8vwLM', np.float64(0.4505)),
 ('n8Q3Z5E', np.float64(0.4604)),
 ('vv6xP53', np.float64(0.6874)),--
 ('ogrV3bl', np.float64(0.4527)),x
 ('ogrV32E', np.float64(0.4598)),
 ('7nMxR02', np.float64(0.5896))]